## Setup

In [1]:
# Load libraries and the API key from .env
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message

# Create the client and choose the model
client = Anthropic()
model = "claude-haiku-4-5"

## Message helpers

In [2]:
# Add a user turn; accepts text, a list of blocks, or a Message
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })

# Add an assistant turn
def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

# Extract only the text from a response
def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

## Chat function with prompt caching

In [3]:
def chat(messages, system=None, temperature=1.0, tools=None):
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        # Copy the list and the last tool, so the originals stay untouched
        tools_clone = tools.copy()
        last_tool = tools_clone[-1].copy()
        # A breakpoint on the last tool caches every tool before it
        last_tool["cache_control"] = {"type": "ephemeral"}
        tools_clone[-1] = last_tool
        params["tools"] = tools_clone

    if system:
        # The system prompt becomes a text block that carries a breakpoint
        params["system"] = [
            {
                "type": "text",
                "text": system,
                "cache_control": {"type": "ephemeral"},
            }
        ]

    return client.messages.create(**params)

## A tool schema and a long system prompt

In [4]:
# A small tool, only here to show how tool caching is configured
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time as a formatted string. Use it when the user asks about today's date or the time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {"type": "string", "description": "Python strftime format codes. Optional."}
        },
        "required": [],
    },
}
tools = [get_current_datetime_schema]

# Build a long system prompt, since short content is never cached
rule = (
    "You are a careful coding assistant. Always explain your reasoning briefly, "
    "prefer simple readable solutions, name variables clearly, handle errors "
    "explicitly, and mention any assumptions you make about the input. "
)
system_prompt = "You are a helpful coding assistant.\n" + (rule * 150)
print("Characters:", len(system_prompt), "(roughly 6,000 tokens)")

Characters: 31986 (roughly 6,000 tokens)


## Helper to read the cache usage

In [5]:
# Print how many tokens were processed normally, written to cache, and read from cache
def show_usage(response):
    u = response.usage
    print("input_tokens (not cached):", u.input_tokens)
    print("cache_creation_input_tokens:", u.cache_creation_input_tokens)
    print("cache_read_input_tokens:", u.cache_read_input_tokens)
    print()

## First request: the cache is written

In [6]:
# First call: the long system prompt (and the tool) are written to the cache
messages = []
add_user_message(messages, "In one sentence, what is a Python list?")

response = chat(messages, system=system_prompt, tools=tools)
show_usage(response)

input_tokens (not cached): 333
cache_creation_input_tokens: 5676
cache_read_input_tokens: 0



## Second request: the cache is read

In [7]:
# Same system prompt and tools, so the cache is read instead of reprocessed
messages = []
add_user_message(messages, "In one sentence, what is a Python dictionary?")

response = chat(messages, system=system_prompt, tools=tools)
show_usage(response)

input_tokens (not cached): 333
cache_creation_input_tokens: 0
cache_read_input_tokens: 5676



## Change one character: the cache is invalidated

In [8]:
# Changing even one character of the system prompt breaks its cache
changed_system = system_prompt + "."

messages = []
add_user_message(messages, "In one sentence, what is a Python tuple?")

response = chat(messages, system=changed_system, tools=tools)
show_usage(response)

input_tokens (not cached): 333
cache_creation_input_tokens: 5676
cache_read_input_tokens: 0



## Why the tools did not give a partial read

In [9]:
# The tool schema alone is far shorter than the minimum cacheable length.
# A breakpoint on content under the minimum writes nothing, so changing the
# system prompt gives a full re-write, not "tools read + system written".
# To see a partial read, make the tool definitions themselves long enough.
print("Tool schema characters:", len(str(get_current_datetime_schema)))

Tool schema characters: 326


This is the caveat from the lesson: tool schemas only produce a separate cache read when they are long on their own, such as several detailed tools.

What to compare

Request 1 has a cache write, request 2 has a cache read, and request 3 has a new write.
Cached reads cost less and respond faster than normal input.
If both cache numbers stay at 0 in every request, the content is probably below the minimum length for your model. Raise the 150 in rule * 150 to 250.